# PowerGraph — Dataset Overview (.tar.gz)

This notebook inspects the **PowerGraph** dataset, introduced in [PowerGraph: A power grid benchmark dataset for graph neural networks](https://arxiv.org/abs/2402.02827).

PowerGraph is a benchmark dataset designed for graph neural networks (GNNs) in power-system modeling. It contains graph-structured datasets for:

- power flow (PF),
- optimal power flow (OPF),
- cascading failure analysis.

The dataset models power grids as graphs, where electrical buses are represented as nodes and transmission lines or transformers are represented as edges.

For the power-flow and optimal-power-flow tasks, PowerGraph uses MATPOWER simulations. For cascading-failure analysis, it uses Cascades, an AC physics-based cascading-failure model. The cascading-failure task includes graph-level labels such as binary failure labels, multiclass labels, demand-not-served regression labels, and explanation masks for cascading edges.

This notebook focuses on the **graph-level cascading-failure data** for the **IEEE24** system.

## Notes

- The datasets reside in `/home/user/data/datasets/PowerGraph/`.
- Available cases currently include:
  - `ieee24.tar.gz`
  - `ieee39.tar.gz`
  - `ieee118.tar.gz`
- This notebook inspects the `ieee24.tar.gz` archive.
- Each `.tar.gz` file contains multiple `.json` files, named as `merged_1.json`, `merged_2.json`, etc.
- Inside the archive, the JSON files follow the structure `ieee24/merged_1.json`.
- Each `.json` file contains multiple graph instances.
- Each graph instance contains:
  - `nodes`: node feature vectors,
  - `edges`: edge feature vectors,
  - `edge_index`: graph connectivity,
  - `labels`: graph-level targets.
- This notebook serves as an EDA for one selected `.json` file.

---
#### Bootstrap

In [1]:
from _notebook_bootstrap import bootstrap

repo_root, datasets_root = bootstrap()
repo_root, datasets_root

(PosixPath('/home/user/workspace/datasets-utilities'),
 PosixPath('/home/user/data/datasets'))

---
#### Imports

In [2]:
import json
import tarfile
from pathlib import Path
import pandas as pd
import numpy as np

---
#### Load dataset object

In [3]:
# Select your file (can be changed to any other snapshot path under datasets_root)
relpath = "PowerGraph/ieee24.tar.gz"

datasets_root = Path(datasets_root)
archive_path = datasets_root / relpath

assert archive_path.exists(), f"Archive not found: {archive_path}"

print(f"Archive path: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / (1024 ** 2):,.2f} MB")

Archive path: /home/user/data/datasets/PowerGraph/ieee24.tar.gz
Archive size: 3.03 MB


---
#### Inspect archive contents

In [4]:
with tarfile.open(archive_path, "r:gz") as tar:
    members = [m for m in tar.getmembers() if m.isfile()]

archive_index = pd.DataFrame(
    {
        "name": [m.name for m in members],
        "size_mb": [m.size / (1024 ** 2) for m in members],
    }
)

json_members = archive_index.loc[
    archive_index["name"].str.endswith(".json"), "name"
].tolist()

display(archive_index.head())

print(f"Files in archive: {len(archive_index)}")
print(f"JSON files in archive: {len(json_members)}")

,name,size_mb
0,ieee24/merged_19.json,0.818534
1,ieee24/merged_106.json,0.818608
2,ieee24/merged_161.json,0.819035
3,ieee24/merged_17.json,0.818342
4,ieee24/merged_126.json,0.819950


Files in archive: 215
JSON files in archive: 215


---
#### Select JSON snapshot

In [5]:
# Select one JSON file inside the archive
target = json_members[0]

print(f"Selected JSON file: {target}")

Selected JSON file: ieee24/merged_19.json


---
#### Load JSON Snapshot

In [6]:
with tarfile.open(archive_path, "r:gz") as tar:
    with tar.extractfile(target) as f:
        data = json.load(f)

print(type(data))
print(f"Number of OPF examples in this JSON file: {len(data)}")

first_id = next(iter(data))
first_example = data[first_id]

print(f"First example ID: {first_id}")

<class 'dict'>
Number of OPF examples in this JSON file: 100
First example ID: 1800


---
#### JSON structure helper

In [7]:
def print_json_structure(obj, indent=0, max_depth=6):
    prefix = "  " * indent

    if indent > max_depth:
        print(prefix + "...")
        return

    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, dict):
                print(f"{prefix}{key}: dict")
                print_json_structure(value, indent + 1, max_depth)

            elif isinstance(value, list):
                print(f"{prefix}{key}: list[{len(value)}]")
                if len(value) > 0:
                    print_json_structure(value[0], indent + 1, max_depth)

            else:
                print(f"{prefix}{key}: {type(value).__name__}")

    elif isinstance(obj, list):
        print(f"{prefix}list[{len(obj)}]")
        if len(obj) > 0:
            print_json_structure(obj[0], indent + 1, max_depth)

    else:
        print(f"{prefix}{type(obj).__name__}")

---
#### Inspect 1st example structure

In [8]:
print_json_structure({first_id: first_example}, max_depth=6)

1800: dict
  nodes: dict
    1: list[24]
      float
    2: list[24]
      float
    3: list[24]
      float
  edges: dict
    1: list[38]
      float
    2: list[38]
      float
    3: list[38]
      float
    4: list[38]
      float
    5: list[38]
      float
    6: list[38]
      float
    7: list[38]
      float
    8: list[38]
      float
  edge_index: dict
    1: list[38]
      float
    2: list[38]
      float
  labels: dict
    1: float
    2: list[4]
      float
    3: float
    4: NoneType


#### Feature meaning

Each PowerGraph JSON file contains multiple graph instances. Each instance corresponds to one pre-outage operating condition and one outage scenario.

Each instance has four main objects:

- `nodes`: node-level input features.
- `edges`: edge-level input features.
- `edge_index`: graph connectivity.
- `labels`: graph-level outputs.

For the graph-level cascading-failure task:

- Nodes correspond to buses.
- Edges correspond to branches, i.e. transmission lines and transformers.
- The graph label depends on the cascading-failure outcome.

The PowerGraph paper describes three node features:

1. Net active power
2. Net apparent power
3. Voltage magnitude

It describes four edge features:

1. Active power flow
2. Reactive power flow
3. Line reactance
4. Line rating

In this JSON export, `edges` contains 8 feature vectors. The first four are named according to the documented PowerGraph features. The remaining four are kept as generic extra edge features until their exact meaning is verified from the dataset conversion code or raw source files.

---
#### Define PowerGraph feature names

In [9]:
POWERGRAPH_FEATURE_NAMES = {
    "nodes": {
        "1": "p_net",
        "2": "s_net",
        "3": "v_mag",
    },

    "edges": {
        "1": "p_flow",
        "2": "q_flow",
        "3": "line_reactance",
        "4": "line_rating",

        # Present in this JSON export, but not part of the four documented edge features.
        "5": "edge_feature_5",
        "6": "edge_feature_6",
        "7": "edge_feature_7",
        "8": "edge_feature_8",
    },

    "edge_index": {
        "1": "sender",
        "2": "receiver",
    },

    "labels": {
        "1": "binary_label",
        "2": "multiclass_label",
        "3": "regression_label_dns",
        "4": "explanation_mask",
    },
}

---
#### Define short feature descriptions

In [10]:
POWERGRAPH_FEATURE_DESCRIPTIONS = {
    "p_net": "Net active power at bus",
    "s_net": "Net apparent power at bus",
    "v_mag": "Voltage magnitude at bus",

    "p_flow": "Active power flow on branch",
    "q_flow": "Reactive power flow on branch",
    "line_reactance": "Branch / line reactance",
    "line_rating": "Branch / line thermal rating",

    "edge_feature_5": "Extra edge feature in this JSON export; not documented in the four-feature PowerGraph graph-level schema",
    "edge_feature_6": "Extra edge feature in this JSON export; not documented in the four-feature PowerGraph graph-level schema",
    "edge_feature_7": "Extra edge feature in this JSON export; not documented in the four-feature PowerGraph graph-level schema",
    "edge_feature_8": "Extra edge feature in this JSON export; not documented in the four-feature PowerGraph graph-level schema",

    "sender": "Source bus index for each edge",
    "receiver": "Target bus index for each edge",

    "binary_label": "Binary cascading-failure label",
    "multiclass_label": "Four-class cascading-failure label",
    "regression_label_dns": "Regression label: demand not served (DNS)",
    "explanation_mask": "Ground-truth explanation mask for cascading edges; may be None for some instances",
}

---
#### Build feature dictionary table

In [11]:
feature_dictionary_rows = []

for object_type, feature_map in POWERGRAPH_FEATURE_NAMES.items():
    for raw_key, feature_name in feature_map.items():
        feature_dictionary_rows.append(
            {
                "object_type": object_type,
                "raw_key": raw_key,
                "feature": feature_name,
                "description": POWERGRAPH_FEATURE_DESCRIPTIONS.get(feature_name, ""),
            }
        )

feature_dictionary = pd.DataFrame(feature_dictionary_rows)

display(feature_dictionary)

,object_type,raw_key,feature,description
0,nodes,1,p_net,Net active power at bus
1,nodes,2,s_net,Net apparent power at bus
2,nodes,3,v_mag,Voltage magnitude at bus
3,edges,1,p_flow,Active power flow on branch
4,edges,2,q_flow,Reactive power flow on branch
5,edges,3,line_reactance,Branch / line reactance
6,edges,4,line_rating,Branch / line thermal rating
7,edges,5,edge_feature_5,Extra edge feature in this JSON export; not do...
8,edges,6,edge_feature_6,Extra edge feature in this JSON export; not do...
9,edges,7,edge_feature_7,Extra edge feature in this JSON export; not do...


#### Observed structure in the selected example

The next cell checks the observed number of rows and columns for each feature matrix in the selected example.

For topological-perturbation data, the number of AC lines, transformers, or generators may vary across examples because components can be removed.

---
#### Check observed shapes against expected feature names

In [12]:
def get_raw_keyed_block(example, object_type):
    return example[object_type]


def summarize_keyed_block(example, object_type, feature_map):
    block = get_raw_keyed_block(example, object_type)

    rows = []

    for raw_key, feature_name in feature_map.items():
        value = block.get(raw_key)

        if isinstance(value, list):
            value_type = "list"
            length = len(value)
            nested_length = len(value[0]) if length > 0 and isinstance(value[0], list) else None

        elif value is None:
            value_type = "NoneType"
            length = None
            nested_length = None

        else:
            value_type = type(value).__name__
            length = None
            nested_length = None

        rows.append(
            {
                "object_type": object_type,
                "raw_key": raw_key,
                "feature": feature_name,
                "value_type": value_type,
                "length": length,
                "nested_length": nested_length,
                "is_present": raw_key in block,
            }
        )

    return rows


shape_rows = []

for object_type, feature_map in POWERGRAPH_FEATURE_NAMES.items():
    shape_rows.extend(
        summarize_keyed_block(first_example, object_type, feature_map)
    )

shape_summary = pd.DataFrame(shape_rows)

display(shape_summary)

,object_type,raw_key,feature,value_type,length,nested_length,is_present
0,nodes,1,p_net,list,24.0,None,True
1,nodes,2,s_net,list,24.0,None,True
2,nodes,3,v_mag,list,24.0,None,True
3,edges,1,p_flow,list,38.0,None,True
4,edges,2,q_flow,list,38.0,None,True
5,edges,3,line_reactance,list,38.0,None,True
6,edges,4,line_rating,list,38.0,None,True
7,edges,5,edge_feature_5,list,38.0,None,True
8,edges,6,edge_feature_6,list,38.0,None,True
9,edges,7,edge_feature_7,list,38.0,None,True


---
#### Helper for printing example feature tables

In [13]:
def keyed_vectors_to_df(example, object_type, feature_map, index_name):
    block = example[object_type]

    data = {}

    for raw_key, feature_name in feature_map.items():
        if raw_key in block:
            value = block[raw_key]

            if isinstance(value, list):
                data[feature_name] = value

    df = pd.DataFrame(data)
    df.index.name = index_name

    return df.reset_index()


def labels_to_df(example):
    labels = example["labels"]

    rows = []

    for raw_key, feature_name in POWERGRAPH_FEATURE_NAMES["labels"].items():
        value = labels.get(raw_key)

        rows.append(
            {
                "raw_key": raw_key,
                "label": feature_name,
                "value": value,
                "value_type": type(value).__name__,
                "description": POWERGRAPH_FEATURE_DESCRIPTIONS.get(feature_name, ""),
            }
        )

    return pd.DataFrame(rows)

#### `nodes` examples

The `nodes` block stores feature vectors over buses.

For IEEE24, each node feature vector should have length 24, corresponding to the 24 buses.

In [14]:
nodes_df = keyed_vectors_to_df(
    first_example,
    object_type="nodes",
    feature_map=POWERGRAPH_FEATURE_NAMES["nodes"],
    index_name="bus_index",
)

display(nodes_df.head())
print(nodes_df.shape)

,bus_index,p_net,s_net,v_mag
0,0,-0.014144,-0.034880,-0.199694
1,1,-0.000324,0.048330,-0.211491
2,2,-0.229018,-0.042446,-0.027547
3,3,-0.095838,-0.294985,-0.320790
4,4,-0.092069,-0.301183,-0.005538


(24, 4)


#### `edges` examples

The `edges` block stores feature vectors over branches.

For IEEE24, each edge feature vector should have length 38, corresponding to the 38 branches.

The first four edge features are documented by PowerGraph. This JSON export contains four additional edge feature vectors, which are retained with generic names.

In [15]:
edges_df = keyed_vectors_to_df(
    first_example,
    object_type="edges",
    feature_map=POWERGRAPH_FEATURE_NAMES["edges"],
    index_name="edge_index",
)

display(edges_df.head())
print(edges_df.shape)

,edge_index,p_flow,q_flow,line_reactance,line_rating,edge_feature_5,edge_feature_6,edge_feature_7,edge_feature_8
0,0,0.145879,-0.117304,-0.296049,-0.643725,0.145879,-0.117304,-0.296049,-0.643725
1,1,0.052372,0.031064,0.703951,-0.643725,0.052372,0.031064,0.703951,-0.643725
2,2,0.144458,-0.080408,0.061781,-0.643725,0.144458,-0.080408,0.061781,-0.643725
3,3,0.123151,0.023889,0.275669,-0.643725,0.123151,0.023889,0.275669,-0.643725
4,4,0.142521,-0.090013,0.606637,-0.643725,0.142521,-0.090013,0.606637,-0.643725


(38, 9)


In [16]:
edge_index_df = keyed_vectors_to_df(
    first_example,
    object_type="edge_index",
    feature_map=POWERGRAPH_FEATURE_NAMES["edge_index"],
    index_name="edge_index",
)

# Convert endpoint columns to nullable integers when possible.
for col in ["sender", "receiver"]:
    edge_index_df[col] = pd.to_numeric(edge_index_df[col], errors="coerce").astype("Int64")

display(edge_index_df.head())
print(edge_index_df.shape)

,edge_index,sender,receiver
0,0,1,2
1,1,1,3
2,2,1,5
3,3,2,4
4,4,2,6


(38, 3)


In [17]:
edge_table = edge_index_df.merge(
    edges_df,
    on="edge_index",
    how="left",
)

display(edge_table.head())
print(edge_table.shape)

,edge_index,sender,receiver,p_flow,q_flow,line_reactance,line_rating,edge_feature_5,edge_feature_6,edge_feature_7,edge_feature_8
0,0,1,2,0.145879,-0.117304,-0.296049,-0.643725,0.145879,-0.117304,-0.296049,-0.643725
1,1,1,3,0.052372,0.031064,0.703951,-0.643725,0.052372,0.031064,0.703951,-0.643725
2,2,1,5,0.144458,-0.080408,0.061781,-0.643725,0.144458,-0.080408,0.061781,-0.643725
3,3,2,4,0.123151,0.023889,0.275669,-0.643725,0.123151,0.023889,0.275669,-0.643725
4,4,2,6,0.142521,-0.090013,0.606637,-0.643725,0.142521,-0.090013,0.606637,-0.643725


(38, 11)


#### `labels` examples

The `labels` block stores graph-level outputs.

The expected label fields are:

- `binary_label`: binary classification target.
- `multiclass_label`: four-class classification target.
- `regression_label_dns`: demand-not-served regression target.
- `explanation_mask`: ground-truth cascading-edge explanation mask, when available.

In [18]:
labels_df = labels_to_df(first_example)

display(labels_df)

,raw_key,label,value,value_type,description
0,1,binary_label,0.0,float,Binary cascading-failure label
1,2,multiclass_label,"[0.0, 0.0, 0.0, 1.0]",list,Four-class cascading-failure label
2,3,regression_label_dns,0.0,float,Regression label: demand not served (DNS)
3,4,explanation_mask,None,NoneType,Ground-truth explanation mask for cascading ed...


In [19]:
EDGE_DUPLICATE_PAIRS = [
    ("1", "5"),
    ("2", "6"),
    ("3", "7"),
    ("4", "8"),
]


def compare_edge_feature_pairs(example_id, example, pairs=EDGE_DUPLICATE_PAIRS):
    edges = example["edges"]

    row = {
        "example_id": example_id,
        "n_edges": len(edges["1"]),
    }

    pair_allclose_results = []
    pair_exact_results = []
    pair_max_abs_diffs = []

    for left_key, right_key in pairs:
        left = np.asarray(edges[left_key], dtype=float)
        right = np.asarray(edges[right_key], dtype=float)

        same_shape = left.shape == right.shape
        exact_match = same_shape and np.array_equal(left, right)
        allclose_match = same_shape and np.allclose(left, right, rtol=1e-10, atol=1e-12)

        max_abs_diff = np.max(np.abs(left - right)) if same_shape else np.nan

        pair_name = f"{left_key}_vs_{right_key}"

        row[f"{pair_name}_same_shape"] = same_shape
        row[f"{pair_name}_exact"] = exact_match
        row[f"{pair_name}_allclose"] = allclose_match
        row[f"{pair_name}_max_abs_diff"] = max_abs_diff

        pair_exact_results.append(exact_match)
        pair_allclose_results.append(allclose_match)
        pair_max_abs_diffs.append(max_abs_diff)

    row["all_pairs_exact"] = all(pair_exact_results)
    row["all_pairs_allclose"] = all(pair_allclose_results)
    row["max_abs_diff_all_pairs"] = np.nanmax(pair_max_abs_diffs)

    return row


edge_duplicate_check = pd.DataFrame(
    compare_edge_feature_pairs(example_id, example)
    for example_id, example in data.items()
)

display(edge_duplicate_check.head())
print(edge_duplicate_check.shape)

,example_id,n_edges,1_vs_5_same_shape,1_vs_5_exact,1_vs_5_allclose,1_vs_5_max_abs_diff,2_vs_6_same_shape,2_vs_6_exact,2_vs_6_allclose,2_vs_6_max_abs_diff,...,3_vs_7_exact,3_vs_7_allclose,3_vs_7_max_abs_diff,4_vs_8_same_shape,4_vs_8_exact,4_vs_8_allclose,4_vs_8_max_abs_diff,all_pairs_exact,all_pairs_allclose,max_abs_diff_all_pairs
0,1800,38,True,False,False,0.190796,True,False,False,0.012583,...,False,False,0.257022,True,False,False,0.356275,False,False,0.356275
1,1801,38,True,False,False,0.144458,True,False,False,0.080408,...,False,False,0.061781,True,False,False,0.643725,False,False,0.643725
2,1802,38,True,False,False,0.115807,True,False,False,0.210126,...,False,False,0.058740,True,False,False,0.048583,False,False,0.210126
3,1803,38,True,False,False,0.119612,True,False,False,0.012583,...,False,False,0.257022,True,False,False,0.356275,False,False,0.356275
4,1804,38,True,False,False,0.466755,True,False,False,0.137554,...,False,False,0.249420,True,False,False,0.356275,False,False,0.466755


(100, 21)


In [20]:
duplicate_summary_rows = []

for left_key, right_key in EDGE_DUPLICATE_PAIRS:
    pair_name = f"{left_key}_vs_{right_key}"

    duplicate_summary_rows.append(
        {
            "pair": pair_name,
            "n_examples": len(edge_duplicate_check),
            "same_shape_examples": edge_duplicate_check[f"{pair_name}_same_shape"].sum(),
            "exact_match_examples": edge_duplicate_check[f"{pair_name}_exact"].sum(),
            "allclose_match_examples": edge_duplicate_check[f"{pair_name}_allclose"].sum(),
            "max_abs_diff": edge_duplicate_check[f"{pair_name}_max_abs_diff"].max(),
        }
    )

duplicate_summary = pd.DataFrame(duplicate_summary_rows)

display(duplicate_summary)

print(
    "Examples where all four duplicate pairs match exactly:",
    edge_duplicate_check["all_pairs_exact"].sum(),
    "/",
    len(edge_duplicate_check),
)

print(
    "Examples where all four duplicate pairs match within tolerance:",
    edge_duplicate_check["all_pairs_allclose"].sum(),
    "/",
    len(edge_duplicate_check),
)

print(
    "Maximum absolute difference across all checked pairs:",
    edge_duplicate_check["max_abs_diff_all_pairs"].max(),
)

,pair,n_examples,same_shape_examples,exact_match_examples,allclose_match_examples,max_abs_diff
0,1_vs_5,100,100,0,0,0.560070
1,2_vs_6,100,100,0,0,0.786856
2,3_vs_7,100,100,0,0,0.703951
3,4_vs_8,100,100,0,0,0.643725


Examples where all four duplicate pairs match exactly: 0 / 100
Examples where all four duplicate pairs match within tolerance: 0 / 100
Maximum absolute difference across all checked pairs: 0.7868562627506793


In [21]:
duplicate_failures = edge_duplicate_check.loc[
    ~edge_duplicate_check["all_pairs_allclose"]
]

display(duplicate_failures)

print(f"Number of non-matching examples: {len(duplicate_failures)}")

,example_id,n_edges,1_vs_5_same_shape,1_vs_5_exact,1_vs_5_allclose,1_vs_5_max_abs_diff,2_vs_6_same_shape,2_vs_6_exact,2_vs_6_allclose,2_vs_6_max_abs_diff,...,3_vs_7_exact,3_vs_7_allclose,3_vs_7_max_abs_diff,4_vs_8_same_shape,4_vs_8_exact,4_vs_8_allclose,4_vs_8_max_abs_diff,all_pairs_exact,all_pairs_allclose,max_abs_diff_all_pairs
0,1800,38,True,False,False,0.190796,True,False,False,0.012583,...,False,False,0.257022,True,False,False,0.356275,False,False,0.356275
1,1801,38,True,False,False,0.144458,True,False,False,0.080408,...,False,False,0.061781,True,False,False,0.643725,False,False,0.643725
2,1802,38,True,False,False,0.115807,True,False,False,0.210126,...,False,False,0.058740,True,False,False,0.048583,False,False,0.210126
3,1803,38,True,False,False,0.119612,True,False,False,0.012583,...,False,False,0.257022,True,False,False,0.356275,False,False,0.356275
4,1804,38,True,False,False,0.466755,True,False,False,0.137554,...,False,False,0.249420,True,False,False,0.356275,False,False,0.466755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1895,38,True,False,False,0.060789,True,False,False,0.000167,...,False,False,0.058740,True,False,False,0.048583,False,False,0.060789
96,1896,38,True,False,False,0.380101,True,False,False,0.066498,...,False,False,0.235228,True,False,False,0.356275,False,False,0.380101
97,1897,38,True,False,False,0.198380,True,False,False,0.006990,...,False,False,0.236642,True,False,False,0.643725,False,False,0.643725
98,1898,38,True,False,False,0.010062,True,False,False,0.730154,...,False,False,0.059861,True,False,False,0.643725,False,False,0.730154


Number of non-matching examples: 100


In [22]:
def compare_first_four_to_second_four(example_id, example):
    edges = example["edges"]

    first_four = np.column_stack(
        [np.asarray(edges[str(i)], dtype=float) for i in range(1, 5)]
    )

    second_four = np.column_stack(
        [np.asarray(edges[str(i)], dtype=float) for i in range(5, 9)]
    )

    return {
        "example_id": example_id,
        "same_shape": first_four.shape == second_four.shape,
        "exact_match": np.array_equal(first_four, second_four),
        "allclose_match": np.allclose(first_four, second_four, rtol=1e-10, atol=1e-12),
        "max_abs_diff": np.max(np.abs(first_four - second_four)),
    }


edge_block_duplicate_check = pd.DataFrame(
    compare_first_four_to_second_four(example_id, example)
    for example_id, example in data.items()
)

display(edge_block_duplicate_check.head())

display(edge_block_duplicate_check.describe(include="all"))

print(
    "Full 4-feature edge block duplicated exactly:",
    edge_block_duplicate_check["exact_match"].sum(),
    "/",
    len(edge_block_duplicate_check),
)

print(
    "Full 4-feature edge block duplicated within tolerance:",
    edge_block_duplicate_check["allclose_match"].sum(),
    "/",
    len(edge_block_duplicate_check),
)

,example_id,same_shape,exact_match,allclose_match,max_abs_diff
0,1800,True,False,False,0.356275
1,1801,True,False,False,0.643725
2,1802,True,False,False,0.210126
3,1803,True,False,False,0.356275
4,1804,True,False,False,0.466755


,example_id,same_shape,exact_match,allclose_match,max_abs_diff
count,100,100,100,100,100.000000
unique,100,1,1,1,NaN
top,1800,True,False,False,NaN
freq,1,100,100,100,NaN
mean,NaN,NaN,NaN,NaN,0.440207
std,NaN,NaN,NaN,NaN,0.178575
min,NaN,NaN,NaN,NaN,0.058740
25%,NaN,NaN,NaN,NaN,0.356275
50%,NaN,NaN,NaN,NaN,0.356275
75%,NaN,NaN,NaN,NaN,0.643725


Full 4-feature edge block duplicated exactly: 0 / 100
Full 4-feature edge block duplicated within tolerance: 0 / 100
